Model Tuning and Selection

Objective

Tune baseline models, evaluate performance using ROC-AUC, and select the best candidate model for customer churn prediction. This notebook builds on Notebook 05 and ensures our final model is robust and defensible.



Importing Libraries


In [18]:
# Core libraries
import numpy as np
import pandas as pd


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Model selection
from sklearn.model_selection import GridSearchCV, train_test_split


# Preprocessing
from sklearn.preprocessing import StandardScaler


# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# Metrics
from sklearn.metrics import roc_auc_score


# Reproducibility
RANDOM_STATE = 42

Load Processed Data

In [19]:
df = pd.read_csv('../data/processed/Telco-Customer-Churn-processed.csv')
X = df.drop('Churn', axis=1)
y = df['Churn']


X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

Feature Scaling

In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Hyperparameter Tuning

Logistic Regression

In [21]:
param_grid_lr = {
'C': [0.01, 0.1, 1, 10, 100],
'solver': ['liblinear', 'lbfgs']
}


lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
grid_lr = GridSearchCV(lr, param_grid_lr, cv=5, scoring='roc_auc', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train)


print("Best params:", grid_lr.best_params_)
print("Best ROC-AUC:", grid_lr.best_score_)

Best params: {'C': 10, 'solver': 'liblinear'}
Best ROC-AUC: 0.8459317647527247


Decision Tree

In [22]:
param_grid_dt = {
'max_depth': [3, 5, 7, 10, None],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4]
}


dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
grid_dt = GridSearchCV(dt, param_grid_dt, cv=5, scoring='roc_auc', n_jobs=-1)
grid_dt.fit(X_train, y_train)


print("Best params:", grid_dt.best_params_)
print("Best ROC-AUC:", grid_dt.best_score_)

Best params: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best ROC-AUC: 0.8178597396439005


Random Forest

In [23]:
param_grid_rf = {
'n_estimators': [100, 200],
'max_depth': [None, 5, 7],
'min_samples_split': [2, 5],
'min_samples_leaf': [1, 2]
}


rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1)
grid_rf.fit(X_train, y_train)


print("Best params:", grid_rf.best_params_)
print("Best ROC-AUC:", grid_rf.best_score_)

Best params: {'max_depth': 7, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Best ROC-AUC: 0.8450566875262806


Model Comparison

In [24]:
model_perf = pd.DataFrame({
'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
'Best ROC-AUC': [
grid_lr.best_score_,
grid_dt.best_score_,
grid_rf.best_score_
],
'Best Params': [
grid_lr.best_params_,
grid_dt.best_params_,
grid_rf.best_params_
]
})


model_perf.sort_values(by='Best ROC-AUC', ascending=False)

,Model,Best ROC-AUC,Best Params
0,Logistic Regression,0.845932,"{'C': 10, 'solver': 'liblinear'}"
2,Random Forest,0.845057,"{'max_depth': 7, 'min_samples_leaf': 2, 'min_s..."
1,Decision Tree,0.817860,"{'max_depth': 5, 'min_samples_leaf': 1, 'min_s..."


Final Model Selection

**Best Model:** Logistic Regression (ROC-AUC: 0.846)

**Trade-offs:**
- Logistic Regression: simple, interpretable, fast
- Random Forest: slightly lower ROC-AUC, more complex, harder to explain
- Decision Tree: lowest ROC-AUC, prone to overfitting

**Deployment Justification:**
- Logistic Regression balances performance and interpretability
- Easy to integrate into production pipelines and communicate insights


Save Final Model

In [25]:
import os

# Create the models folder if it doesn't exist
os.makedirs('../models', exist_ok=True)

import joblib

final_model = grid_lr.best_estimator_
joblib.dump(final_model, '../models/logistic_regression_churn_model.pkl')

print("Final model saved successfully!")


Final model saved successfully!


Verify Saved Model

Load the saved Logistic Regression model and test predictions on the test set to ensure correct persistence and performance.


In [26]:
import joblib

# Load the saved model
loaded_model = joblib.load('../models/logistic_regression_churn_model.pkl')

# Test prediction on the first 5 rows of your test set
pred_probs = loaded_model.predict_proba(X_test_scaled[:5])[:, 1]
pred_classes = loaded_model.predict(X_test_scaled[:5])

print("Predicted probabilities:", pred_probs)
print("Predicted classes:", pred_classes)


Predicted probabilities: [0.04758104 0.68253174 0.04898241 0.42562974 0.02321895]
Predicted classes: [0 1 0 0 0]


In [27]:
from sklearn.metrics import roc_auc_score

# Predict probabilities for the positive class
y_pred_probs = loaded_model.predict_proba(X_test_scaled)[:, 1]

# Predict class labels
y_pred_classes = loaded_model.predict(X_test_scaled)

# Calculate ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_probs)
print(f"ROC-AUC on the test set: {roc_auc:.4f}")


ROC-AUC on the test set: 0.8410
